In [ ]:
from pathlib import Path
import sys, json
from IPython.display import display, Image, Markdown
candidates = [Path.cwd() / 'open-set-recognition', Path.cwd(), Path('/kaggle/working/beyond-iid/open-set-recognition')]
TASK_DIR = next((p.resolve() for p in candidates if (p / 'train.py').exists()), None)
if TASK_DIR is None:
    raise FileNotFoundError('Run this notebook from the repository, or update TASK_DIR to the cloned open-set-recognition directory.')
sys.path.insert(0, str(TASK_DIR))
print('Task directory:', TASK_DIR)

In [ ]:
from train import train_all
training = train_all(TASK_DIR, force=False)
display(training['histories'].groupby('method').tail(1))
display(training['selections'])
display(Image(filename=str(TASK_DIR / 'results/training_curves.png')))

In [ ]:
manifest = json.loads((TASK_DIR / 'results/checkpoint_manifest.json').read_text())
display(manifest)
assert manifest['cifar100_used'] is False
assert set(manifest['models']) == {'vanilla', 'gcsc', 'proser'}

In [ ]:
from extract_outputs import extract_all_outputs
extraction = extract_all_outputs(TASK_DIR, download=True, force=False)
display(extraction)
display(json.loads((TASK_DIR / 'results/protocol_lock.json').read_text()))
assert extraction['cifar100_train_used'] is False
assert extraction['near_unknown_count'] == extraction['far_unknown_count'] == 800

In [ ]:
from evaluate_osr import evaluate
evaluation = evaluate(TASK_DIR, download=True)
print('Frozen Vanilla score comparison')
display(evaluation['posthoc'].style.format(precision=4))
print('Training-method comparison')
display(evaluation['trained'].style.format(precision=4))

In [ ]:
display(Image(filename=str(TASK_DIR / 'results/vanilla_score_distributions.png')))
display(Image(filename=str(TASK_DIR / 'results/vanilla_score_agreement.png')))
display(Image(filename=str(TASK_DIR / 'results/vanilla_mls_failure_cases.png')))

In [ ]:
print('Per-class acceptance and dominant known prediction')
display(evaluation['absorption'].style.format({'acceptance_rate': '{:.3f}', 'dominant_prediction_share': '{:.3f}'}))
print('Six Vanilla-MLS failures (three near, three far)')
display(evaluation['failures'])
print('Spearman score agreement')
display(evaluation['correlations'].style.format(precision=3))
print('Largest pairwise score-ranking disagreements')
display(evaluation['disagreements'].head(12))
display(Markdown(evaluation['report_notes']))
print('Saved artifacts:')
for path in sorted((TASK_DIR / 'results').iterdir()):
    if path.is_file(): print(' -', path.name)